In [1]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display
import math

In [2]:
class QuadraticState(TypedDict):
    a: float
    b: float
    c: float
    equation: str
    discriminant: float
    roots: list[float]
    result: str

In [3]:
def show_equation(state: QuadraticState):
    a = state["a"]
    b = state["b"]
    c = state["c"]

    equation = f"{a}x² + ({b})x + ({c}) = 0"

    return {"equation": equation}

In [8]:
def calculate_discriminant(state: QuadraticState):
    a = state["a"]
    b = state["b"]
    c = state["c"]

    d = b**2 - 4*a*c

    return {"discriminant": d}

In [9]:
def two_real_roots(state: QuadraticState):
    a = state["a"]
    b = state["b"]
    d = state["discriminant"]

    root1 = (-b + math.sqrt(d)) / (2*a)
    root2 = (-b - math.sqrt(d)) / (2*a)

    return {
        "roots": [root1, root2],
        "result": "Two distinct real roots"
    }


def one_real_root(state: QuadraticState):
    root = -state["b"] / (2 * state["a"])

    return {
        "roots": [root],
        "result": "One repeated real root"
    }


def no_real_roots(state: QuadraticState):
    return {
        "roots": [],
        "result": "No real roots"
    }

In [10]:
def check_discriminant(state: QuadraticState) -> Literal["two_roots", "one_root", "no_roots"]:
    d = state["discriminant"]

    if d > 0:
        return "two_roots"
    elif d == 0:
        return "one_root"
    else:
        return "no_roots"

In [11]:
graph = StateGraph(QuadraticState)

graph.add_node("show_equation", show_equation)
graph.add_node("calculate_discriminant", calculate_discriminant)
graph.add_node("two_roots", two_real_roots)
graph.add_node("one_root", one_real_root)
graph.add_node("no_roots", no_real_roots)

graph.add_edge(START, "show_equation")
graph.add_edge("show_equation", "calculate_discriminant")

graph.add_conditional_edges(
    "calculate_discriminant",
    check_discriminant
)

graph.add_edge("two_roots", END)
graph.add_edge("one_root", END)
graph.add_edge("no_roots", END)

workflow = graph.compile()

In [12]:
initial_state = {"a": 1, "b": -3, "c": 2}

final_state = workflow.invoke(initial_state)

print("Equation:", final_state["equation"])
print("Discriminant:", final_state["discriminant"])
print("Result:", final_state["result"])
print("Roots:", final_state["roots"])

Equation: 1x² + (-3)x + (2) = 0
Discriminant: 1
Result: Two distinct real roots
Roots: [2.0, 1.0]
